# Assignment 04 · Notebook 03
# Mạng nơ-ron tích chập một chiều bằng TensorFlow/Keras và tổng hợp so sánh ba framework


| Mục | Nội dung |
|---|---|
| Học phần | Phát triển các Hệ thống Thông minh |
| Cơ sở đào tạo | Học viện Công nghệ Bưu chính Viễn thông |
| Sinh viên | **Nguyễn Duy Nghĩa** |
| Mã sinh viên | **B23DCCN600** |
| Lớp | **D23CTPM01** |
| Giảng viên hướng dẫn | **PGS.TS Trần Đình Quế** |
| Học kỳ | Học kỳ 1 năm học 2026 - 2027 |
| Assignment | 04 - Convolutional Neural Networks |
| Miền dữ liệu | `house_price` (hồi quy giá bất động sản Hoa Kỳ) |


---

## Mục tiêu của notebook

Notebook này khép lại miền `house_price` với hai nhiệm vụ tách bạch.

**Phần A, hiện thực TensorFlow/Keras.** Tái lập đúng kiến trúc của hợp đồng bằng API `Sequential`,
với `layers.Conv1D(padding='same')`, `layers.MaxPooling1D`, `layers.Dense` và hàm mất mát
`losses.MeanSquaredError`. Phần này cũng làm rõ khác biệt quy ước quan trọng nhất giữa Keras và
PyTorch: Keras dùng bố cục **kênh sau cùng** $(N, L, C)$ trong khi PyTorch dùng **kênh trước**
$(N, C, L)$.

**Phần B, tổng hợp.** Đọc ba file trung gian `_partial_numpy.json`, `_partial_pytorch.json`,
`_partial_tensorflow.json`, dựng ba hình tổng hợp còn lại của miền theo Mục 6 của hợp đồng, và ghi
file kết quả cuối cùng `metrics_house_price.json` theo đúng schema hồi quy ở Mục 5.3.

Điều kiện tiên quyết: notebook 01 và notebook 02 phải đã được chạy xong, vì hai file trung gian của
chúng là đầu vào bắt buộc của phần B.

---

## 1. Nhập thư viện và cố định hạt giống ngẫu nhiên

TensorFlow ghi khá nhiều thông báo khởi tạo ra luồng lỗi chuẩn. Biến môi trường
`TF_CPP_MIN_LOG_LEVEL` được đặt trước khi nạp thư viện để giữ nhật ký notebook gọn gàng, đây chỉ là
thay đổi về mức độ chi tiết của log chứ không ảnh hưởng tới kết quả tính toán.

In [1]:
# ====== Thư viện chuẩn của báo cáo ======
import os, json, time, math, warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")                      # backend không cần màn hình, phù hợp nbconvert
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")

# ====== Hạt giống ngẫu nhiên: cố định để mọi lần chạy tái lập được ======
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# ====== Quy ước vẽ hình theo hợp đồng tích hợp (Mục 5.2) ======
plt.rcParams["font.sans-serif"] = ["Segoe UI", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["figure.facecolor"] = "white"
plt.rcParams["savefig.facecolor"] = "white"
plt.rcParams["figure.dpi"] = 110
sns.set_style("whitegrid")

# ====== Đường dẫn tương đối tính từ thư mục notebooks/ ======
DATA_PATH = "../data/usa_real_estate_150k.csv"
FIG_DIR   = "../reports/figures"
REP_DIR   = "../reports"
os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(REP_DIR, exist_ok=True)

print("NumPy      :", np.__version__)
print("pandas     :", pd.__version__)
print("matplotlib :", matplotlib.__version__)
print("seaborn    :", sns.__version__)
print("RANDOM_SEED:", RANDOM_SEED)

NumPy      : 2.4.0
pandas     : 2.3.3
matplotlib : 3.10.8
seaborn    : 0.13.2
RANDOM_SEED: 42


In [2]:
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# Giới hạn số luồng vì cùng lý do đã nêu ở notebook 02: với mô hình 1 377 tham số, chi phí
# đồng bộ giữa hàng chục luồng lớn hơn chính phép tính.
TF_THREADS = min(4, os.cpu_count() or 4)
tf.config.threading.set_intra_op_parallelism_threads(TF_THREADS)
tf.config.threading.set_inter_op_parallelism_threads(TF_THREADS)

tf.random.set_seed(RANDOM_SEED)
keras.utils.set_random_seed(RANDOM_SEED)

print("TensorFlow :", tf.__version__)
print("Keras      :", keras.__version__)
print("GPU khả dụng:", tf.config.list_physical_devices("GPU"))
print("Thiết bị dùng: CPU (theo hợp đồng Mục 1)")
print("Số luồng intra-op:", tf.config.threading.get_intra_op_parallelism_threads())

TensorFlow : 2.21.0
Keras      : 3.15.1


GPU khả dụng: []
Thiết bị dùng: CPU (theo hợp đồng Mục 1)
Số luồng intra-op: 4


---

## 2. Nạp dữ liệu và tiền xử lý

Khối lệnh giữ nguyên tuyệt đối so với notebook 01 và 02, kể cả bước winsorize ở ngưỡng năm độ lệch
chuẩn. Đây là điều kiện để phép so sánh ba framework ở phần B là so sánh hiện thực chứ không phải
so sánh dữ liệu.

In [3]:
# ---------- 1. Nạp dữ liệu thô ----------
raw = pd.read_csv(DATA_PATH)
n_raw = len(raw)

# ---------- 2. Loại bản ghi thiếu bốn trường cốt lõi ----------
df = raw.dropna(subset=["price", "house_size", "bed", "bath"]).copy()

# ---------- 3. Lọc khoảng giá trị hợp lệ theo hợp đồng ----------
df = df[(df["price"] >= 10_000) & (df["price"] <= 5_000_000)]
df = df[(df["house_size"] >= 200) & (df["house_size"] <= 20_000)]

# ---------- 4. Xử lý acre_lot thiếu: điền trung vị rồi chặn dưới để lấy log an toàn ----------
ACRE_MEDIAN = df["acre_lot"].median()
df["acre_lot"] = df["acre_lot"].fillna(ACRE_MEDIAN).clip(lower=1e-3)

# ---------- 5. Kỹ thuật đặc trưng: đúng 8 đặc trưng, đúng thứ tự hợp đồng ----------
df["log_house_size"] = np.log(df["house_size"])
df["total_rooms"]    = df["bed"] + df["bath"]
df["bed_bath_prod"]  = df["bed"] * df["bath"]
df["sqft_per_room"]  = df["house_size"] / df["total_rooms"].replace(0, np.nan)
df["bath_bed_ratio"] = df["bath"] / df["bed"].replace(0, np.nan)
df["log_acre_lot"]   = np.log(df["acre_lot"])

# ---------- 6. Mục tiêu hồi quy trên thang logarit ----------
df["log_price"] = np.log(df["price"])

FEATURES = ["log_house_size", "bed", "bath", "total_rooms",
            "bed_bath_prod", "sqft_per_room", "bath_bed_ratio", "log_acre_lot"]
TARGET   = "log_price"

df = df.dropna(subset=FEATURES + [TARGET])
n_clean = len(df)

print(f"Số bản ghi thô      n_raw   = {n_raw:,}")
print(f"Số bản ghi sạch     n_clean = {n_clean:,}")
print(f"Tỷ lệ giữ lại               = {n_clean / n_raw:.4f}")
print(f"Trung vị acre_lot dùng để điền khuyết = {ACRE_MEDIAN}")
print()
print("Thống kê mô tả 8 đặc trưng:")
display(df[FEATURES].describe().T[["mean", "std", "min", "25%", "50%", "75%", "max"]].round(4))

Số bản ghi thô      n_raw   = 150,000
Số bản ghi sạch     n_clean = 150,000
Tỷ lệ giữ lại               = 1.0000
Trung vị acre_lot dùng để điền khuyết = 0.23

Thống kê mô tả 8 đặc trưng:


,mean,std,min,25%,50%,75%,max
log_house_size,7.4718,0.5244,5.2983,7.1066,7.4478,7.8038,9.6158
bed,3.3275,1.4828,1.0000,3.0000,3.0000,4.0000,47.0000
bath,2.5015,1.2975,1.0000,2.0000,2.0000,3.0000,39.0000
total_rooms,5.8290,2.5063,2.0000,4.0000,5.0000,7.0000,86.0000
bed_bath_prod,9.5235,12.0217,1.0000,4.0000,6.0000,12.0000,1833.0000
sqft_per_room,342.7944,117.9659,29.1429,272.0000,323.0000,389.4000,6500.0000
bath_bed_ratio,0.7910,0.3193,0.0870,0.5000,0.7500,1.0000,10.0000
log_acre_lot,-1.3090,1.4574,-6.9078,-2.0402,-1.4697,-0.7765,11.5129


In [4]:
X_all = df[FEATURES].to_numpy(dtype=np.float64)
y_all = df[TARGET].to_numpy(dtype=np.float64)

# Tách test trước (20%), rồi tách validation từ phần còn lại (20% của 80% = 16% tổng thể)
X_tmp, X_test, y_tmp, y_test = train_test_split(
    X_all, y_all, test_size=0.20, random_state=RANDOM_SEED)
X_train, X_val, y_train, y_val = train_test_split(
    X_tmp, y_tmp, test_size=0.20, random_state=RANDOM_SEED)

# Chuẩn hóa: fit CHỈ trên train, sau đó transform cho val và test
scaler = StandardScaler().fit(X_train)

# Winsorize sau chuẩn hóa: chặn mọi tọa độ trong khoảng +/- CLIP_SIGMA độ lệch chuẩn.
# Lý do: một số bản ghi có bed_bath_prod tới 1833 (hơn 160 độ lệch chuẩn). Qua hai tầng
# tích chập tuyến tính cộng ReLU, giá trị đó tạo ra dự đoán log_price rất lớn, và sau khi
# lấy np.exp thì sai số USD của vài chục bản ghi lấn át toàn bộ 30 000 mẫu kiểm tra.
CLIP_SIGMA = 5.0
X_train_s = np.clip(scaler.transform(X_train), -CLIP_SIGMA, CLIP_SIGMA)
X_val_s   = np.clip(scaler.transform(X_val),   -CLIP_SIGMA, CLIP_SIGMA)
X_test_s  = np.clip(scaler.transform(X_test),  -CLIP_SIGMA, CLIP_SIGMA)
n_clip_train = int((np.abs(scaler.transform(X_train)) > CLIP_SIGMA).any(axis=1).sum())

# Định dạng chuỗi cho CNN 1 chiều: (N, C_in = 1, L = 8)
Xtr = X_train_s.reshape(-1, 1, 8).astype(np.float32)
Xva = X_val_s.reshape(-1, 1, 8).astype(np.float32)
Xte = X_test_s.reshape(-1, 1, 8).astype(np.float32)
ytr = y_train.reshape(-1, 1).astype(np.float32)
yva = y_val.reshape(-1, 1).astype(np.float32)
yte = y_test.reshape(-1, 1).astype(np.float32)

n_train, n_val, n_test = len(Xtr), len(Xva), len(Xte)
print(f"Train      : {n_train:,} mẫu  ->  tensor {Xtr.shape}")
print(f"Validation : {n_val:,} mẫu  ->  tensor {Xva.shape}")
print(f"Test       : {n_test:,} mẫu  ->  tensor {Xte.shape}")
print()
print("Trung bình sau chuẩn hóa trên train (kỳ vọng xấp xỉ 0):")
print(np.round(Xtr.reshape(-1, 8).mean(axis=0), 6))
print("Độ lệch chuẩn sau chuẩn hóa trên train (kỳ vọng xấp xỉ 1):")
print(np.round(Xtr.reshape(-1, 8).std(axis=0), 6))
print()
print(f"Số mẫu train bị winsorize ở ít nhất một tọa độ: {n_clip_train:,}"
      f"  ({n_clip_train / n_train * 100:.3f}% tập huấn luyện)")
print()
print(f"log_price train: mean = {ytr.mean():.4f}, std = {ytr.std():.4f}")
print(f"Giá tương ứng exp(mean) = {np.exp(ytr.mean()):,.0f} USD")

Train      : 96,000 mẫu  ->  tensor (96000, 1, 8)
Validation : 24,000 mẫu  ->  tensor (24000, 1, 8)
Test       : 30,000 mẫu  ->  tensor (30000, 1, 8)

Trung bình sau chuẩn hóa trên train (kỳ vọng xấp xỉ 0):
[-0.       -0.00681  -0.003464 -0.004927 -0.015516 -0.009928 -0.001579
 -0.001593]
Độ lệch chuẩn sau chuẩn hóa trên train (kỳ vọng xấp xỉ 1):
[1.000008 0.948343 0.973409 0.961575 0.755254 0.899185 0.986466 0.989924]

Số mẫu train bị winsorize ở ít nhất một tọa độ: 974  (1.015% tập huấn luyện)

log_price train: mean = 12.9033, std = 0.8815
Giá tương ứng exp(mean) = 401,652 USD


In [5]:
def danh_gia_hoi_quy(y_true_log, y_pred_log):
    """Tính đủ bộ chỉ số hồi quy theo hợp đồng Mục 5.3.

    Tham số đầu vào nằm trên THANG LOG. Sai số USD được quy đổi ngược bằng exp().
    """
    yt = np.asarray(y_true_log, dtype=np.float64).ravel()
    yp = np.asarray(y_pred_log, dtype=np.float64).ravel()

    # --- Thang log ---
    err_log  = yp - yt
    rmse_log = float(np.sqrt(np.mean(err_log ** 2)))
    mae_log  = float(np.mean(np.abs(err_log)))
    ss_res   = float(np.sum(err_log ** 2))
    ss_tot   = float(np.sum((yt - yt.mean()) ** 2))
    r2       = float(1.0 - ss_res / ss_tot)

    # --- Quy đổi ngược về USD ---
    yt_usd = np.exp(yt)
    yp_usd = np.exp(yp)
    err_usd  = yp_usd - yt_usd
    rmse_usd = float(np.sqrt(np.mean(err_usd ** 2)))
    mae_usd  = float(np.mean(np.abs(err_usd)))

    return {"rmse_usd": rmse_usd, "mae_usd": mae_usd, "r2": r2,
            "rmse_log": rmse_log, "mae_log": mae_log}


def lay_scatter_sample(y_true_log, y_pred_log, n=200, seed=RANDOM_SEED):
    """Rút 200 điểm ngẫu nhiên (thang log) để hợp đồng dựng biểu đồ tán xạ."""
    rng = np.random.default_rng(seed)
    yt = np.asarray(y_true_log, dtype=np.float64).ravel()
    yp = np.asarray(y_pred_log, dtype=np.float64).ravel()
    idx = rng.choice(len(yt), size=min(n, len(yt)), replace=False)
    return {"y_true": yt[idx].tolist(), "y_pred": yp[idx].tolist()}


print("Đã định nghĩa hai hàm dùng chung: danh_gia_hoi_quy() và lay_scatter_sample().")
print("Bộ chỉ số: rmse_usd, mae_usd, r2, rmse_log, mae_log (theo CONTRACT Mục 5.3).")

Đã định nghĩa hai hàm dùng chung: danh_gia_hoi_quy() và lay_scatter_sample().
Bộ chỉ số: rmse_usd, mae_usd, r2, rmse_log, mae_log (theo CONTRACT Mục 5.3).


---

## 3. Khác biệt quy ước giữa Keras và PyTorch

### 3.1 Bố cục tensor: kênh sau cùng so với kênh trước

Đây là nguồn lỗi phổ biến nhất khi chuyển mã giữa hai thư viện.

| Thư viện | Bố cục mặc định | Hình dạng đầu vào của bài toán này |
|---|---|---|
| PyTorch `nn.Conv1d` | kênh trước, `channels_first` | $(N,\; 1,\; 8)$ |
| Keras `layers.Conv1D` | kênh sau cùng, `channels_last` | $(N,\; 8,\; 1)$ |

Hai bố cục mô tả **cùng một dữ liệu**, chỉ khác thứ tự trục. Nếu đưa nhầm tensor $(N, 1, 8)$ vào
Keras, thư viện sẽ diễn giải nó là một chuỗi dài 1 với 8 kênh. Khi đó tích chập $K=3$ trên chuỗi
dài 1 vẫn chạy được nhờ đệm `same`, không báo lỗi nào, nhưng mô hình học một bài toán hoàn toàn
khác và kết quả sẽ tệ một cách khó hiểu. Vì vậy notebook này hoán vị trục một cách tường minh bằng
`np.transpose(X, (0, 2, 1))` và in ra hình dạng để kiểm chứng.

### 3.2 Bộ trọng số

Trọng số của `layers.Conv1D` có hình dạng $(K,\; C_{\text{in}},\; C_{\text{out}})$, trong khi
PyTorch dùng $(C_{\text{out}},\; C_{\text{in}},\; K)$. Tổng số phần tử giống nhau, chỉ khác cách sắp
xếp, nên số tham số của hai mô hình vẫn phải trùng khớp bằng 1 377.

### 3.3 Khởi tạo mặc định

Keras khởi tạo `glorot_uniform`, tức phân phối đều với biên
$\sqrt{6 / (\text{fan\_in} + \text{fan\_out})}$; PyTorch dùng Kaiming Uniform; notebook 01 dùng
He Normal. Ba cách khác nhau nhưng cùng một họ tỷ lệ theo $1/\sqrt{\text{fan}}$. Như đã nêu ở
notebook 02 mục 3.3, báo cáo cố ý giữ nguyên mặc định của từng framework.

In [6]:
# Hoán vị trục: (N, C=1, L=8) -> (N, L=8, C=1) cho bố cục kênh sau cùng của Keras
Xtr_k = np.transpose(Xtr, (0, 2, 1)).astype(np.float32)
Xva_k = np.transpose(Xva, (0, 2, 1)).astype(np.float32)
Xte_k = np.transpose(Xte, (0, 2, 1)).astype(np.float32)

print("Hình dạng cho PyTorch (kênh trước)    :", Xtr.shape)
print("Hình dạng cho Keras  (kênh sau cùng)  :", Xtr_k.shape)
print()
print("Kiểm tra bảo toàn dữ liệu trên mẫu đầu tiên:")
print("  PyTorch Xtr[0, 0, :] =", np.round(Xtr[0, 0, :], 4))
print("  Keras   Xtr_k[0, :, 0] =", np.round(Xtr_k[0, :, 0], 4))
print("  Trùng khớp:", np.allclose(Xtr[0, 0, :], Xtr_k[0, :, 0]))

Hình dạng cho PyTorch (kênh trước)    : (96000, 1, 8)
Hình dạng cho Keras  (kênh sau cùng)  : (96000, 8, 1)

Kiểm tra bảo toàn dữ liệu trên mẫu đầu tiên:
  PyTorch Xtr[0, 0, :] = [-0.0163 -0.8969  1.1605  0.0713 -0.1317 -0.4517  3.7892 -0.1097]
  Keras   Xtr_k[0, :, 0] = [-0.0163 -0.8969  1.1605  0.0713 -0.1317 -0.4517  3.7892 -0.1097]
  Trùng khớp: True


---

## 4. Kiểm chứng ngữ nghĩa: lặp lại ví dụ tính tay lần thứ ba

Notebook 01 tính tay và notebook 02 đã kiểm chứng với PyTorch: với $x = [1, 2, 3, 4]$,
$w = [1, 0, -1]$, $b = 0{,}5$ và đệm `same`, kết quả phải là
$y = [-1{,}5,\; -1{,}5,\; -1{,}5,\; 3{,}5]$.

Kiểm chứng lần thứ ba trên Keras hoàn tất bộ ba: nếu cả ba hiện thực cho cùng một dãy số, thì mọi
chênh lệch kết quả huấn luyện chắc chắn không đến từ cách hiểu phép tích chập.

In [7]:
conv_kiem_chung = layers.Conv1D(filters=1, kernel_size=3, padding="same", use_bias=True)
mo_hinh_kiem_chung = keras.Sequential([layers.Input(shape=(4, 1)), conv_kiem_chung])

# Trọng số Keras có hình dạng (K, C_in, C_out) = (3, 1, 1)
W_keras = np.array([[[1.0]], [[0.0]], [[-1.0]]], dtype=np.float32)
b_keras = np.array([0.5], dtype=np.float32)
conv_kiem_chung.set_weights([W_keras, b_keras])

x_demo_k = np.array([[[1.0], [2.0], [3.0], [4.0]]], dtype=np.float32)   # (1, L=4, C=1)
y_keras = mo_hinh_kiem_chung.predict(x_demo_k, verbose=0)
y_hand = np.array([[-1.5], [-1.5], [-1.5], [3.5]], dtype=np.float32)

print("Hình dạng trọng số Keras   :", W_keras.shape, "= (K, C_in, C_out)")
print("Chuỗi vào                  :", x_demo_k.ravel())
print("Kết quả layers.Conv1D      :", y_keras.ravel())
print("Kết quả tính tay (NB01)    :", y_hand.ravel())
print("Sai lệch tuyệt đối lớn nhất:", float(np.max(np.abs(y_keras.ravel() - y_hand.ravel()))))
assert np.allclose(y_keras.ravel(), y_hand.ravel(), atol=1e-6)
print()
print("KẾT LUẬN: cả ba hiện thực (NumPy thuần, PyTorch, Keras) đồng nhất về ngữ nghĩa tích chập.")

Hình dạng trọng số Keras   : (3, 1, 1) = (K, C_in, C_out)
Chuỗi vào                  : [1. 2. 3. 4.]
Kết quả layers.Conv1D      : [-1.5 -1.5 -1.5  3.5]
Kết quả tính tay (NB01)    : [-1.5 -1.5 -1.5  3.5]
Sai lệch tuyệt đối lớn nhất: 0.0

KẾT LUẬN: cả ba hiện thực (NumPy thuần, PyTorch, Keras) đồng nhất về ngữ nghĩa tích chập.


### Diễn giải kiểm chứng

Sai lệch tuyệt đối lớn nhất bằng 0,0. Cùng với hai kiểm chứng trước đó, kết quả này khẳng định ba
điểm chung của cả ba hiện thực: đều thực hiện tương quan chéo chứ không lật nhân, đều đệm không đối
xứng hai bên, và đều cộng chệch theo trục kênh đầu ra.

Một chi tiết đáng chú ý là hình dạng trọng số $(3, 1, 1)$ của Keras so với $(1, 1, 3)$ của PyTorch.
Sự khác nhau này thuần túy về cách lưu trữ; nếu cần chuyển trọng số giữa hai thư viện chỉ cần hoán
vị trục theo quy tắc $(K, C_{\text{in}}, C_{\text{out}}) \leftrightarrow (C_{\text{out}}, C_{\text{in}}, K)$.

---

## 5. Định nghĩa mô hình Keras

Kiến trúc bám sát hợp đồng, với đầu ra tuyến tính và mất mát MSE:

$$
(8, 1) \;\rightarrow\; \text{Conv1D}(16, K{=}3, \text{same}) \;\rightarrow\; \text{ReLU}
  \;\rightarrow\; \text{Conv1D}(16, K{=}3, \text{same}) \;\rightarrow\; \text{ReLU}
  \;\rightarrow\; \text{MaxPooling1D}(2) \;\rightarrow\; \text{Flatten}
  \;\rightarrow\; \text{Dense}(8) \;\rightarrow\; \text{ReLU} \;\rightarrow\; \text{Dense}(1)
$$

Tầng cuối được khai báo tường minh là `layers.Dense(1, activation=None)`. Trong Keras, giá trị mặc
định của `activation` đã là `None`, nhưng báo cáo vẫn ghi rõ để nhấn mạnh rằng đây là một **quyết
định thiết kế có chủ đích** xuất phát từ bản chất hồi quy của bài toán, không phải kết quả của việc
bỏ quên tham số.

In [8]:
keras.utils.set_random_seed(RANDOM_SEED)

model_k = keras.Sequential([
    layers.Input(shape=(8, 1), name="chuoi_8_dac_trung"),
    layers.Conv1D(16, kernel_size=3, padding="same", activation=None, name="conv1"),
    layers.ReLU(name="relu1"),
    layers.Conv1D(16, kernel_size=3, padding="same", activation=None, name="conv2"),
    layers.ReLU(name="relu2"),
    layers.MaxPooling1D(pool_size=2, name="maxpool"),
    layers.Flatten(name="flatten"),
    layers.Dense(8, activation=None, name="dense1"),
    layers.ReLU(name="relu3"),
    layers.Dense(1, activation=None, name="dau_ra_tuyen_tinh"),
], name="cnn1d_hoi_quy_gia_nha")

model_k.compile(optimizer=keras.optimizers.Adam(learning_rate=3e-3),
                loss=keras.losses.MeanSquaredError(),
                metrics=[keras.metrics.MeanAbsoluteError(name="mae")])

model_k.summary()
params_k = int(sum(np.prod(w.shape) for w in model_k.trainable_weights))
print()
print("Tổng tham số Keras   :", f"{params_k:,}")
print("Tổng tham số NumPy   : 1,377")
print("Tổng tham số PyTorch : 1,377")
print("Ba hiện thực trùng khớp:", params_k == 1377)

Model: "cnn1d_hoi_quy_gia_nha"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1 (Conv1D)                  │ (None, 8, 16)          │            64 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ relu1 (ReLU)                    │ (None, 8, 16)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2 (Conv1D)                  │ (None, 8, 16)          │           784 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ relu2 (ReLU)                    │ (None, 8, 16)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ maxpool (MaxPooling1D)          │ (None, 4, 16)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense1 (Dense)                  │ (None, 8)              │           520 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ relu3 (ReLU)                    │ (None, 8)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dau_ra_tuyen_tinh (Dense)       │ (None, 1)              │             9 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,377 (5.38 KB)

 Trainable params: 1,377 (5.38 KB)

 Non-trainable params: 0 (0.00 B)


Tổng tham số Keras   : 1,377
Tổng tham số NumPy   : 1,377
Tổng tham số PyTorch : 1,377
Ba hiện thực trùng khớp: True


### Diễn giải bảng tóm tắt mô hình

Bảng `summary()` xác nhận chuỗi biến đổi hình dạng đúng như thiết kế: đầu vào $(8, 1)$ giữ nguyên
độ dài 8 qua hai tầng tích chập nhờ đệm `same`, chỉ tăng số kênh lên 16; `MaxPooling1D(2)` nén
xuống $(4, 16)$; `Flatten` cho véc-tơ 64 chiều; hai tầng `Dense` đưa về 8 rồi 1.

Tổng tham số 1 377 trùng khớp với cả NumPy lẫn PyTorch. Ba hiện thực độc lập cùng cho một con số là
bằng chứng mạnh rằng cả ba đã hiểu kiến trúc của hợp đồng theo cùng một cách.

---

## 6. Huấn luyện

Cấu hình giữ nguyên: Adam với tốc độ học $3 \times 10^{-3}$, kích thước lô 256, 40 epoch. Keras
nhận tập kiểm định trực tiếp qua tham số `validation_data` và tự tính mất mát kiểm định sau mỗi
epoch.

Callback `ModelCheckpoint` được dùng để lưu bộ trọng số tốt nhất theo `val_loss`. Cơ chế này tương
đương với đoạn mã thủ công đã viết ở hai notebook trước, nhưng gọn hơn nhiều và là cách làm chuẩn
trong Keras.

In [9]:
EPOCHS_K = 40
BATCH_K  = 256

ckpt_path = "../models/house_price_cnn1d_keras.weights.h5"
os.makedirs("../models", exist_ok=True)

cb_ckpt = keras.callbacks.ModelCheckpoint(
    filepath=ckpt_path, monitor="val_loss", mode="min",
    save_best_only=True, save_weights_only=True, verbose=0)

t0 = time.time()
lich_su = model_k.fit(
    Xtr_k, ytr,
    validation_data=(Xva_k, yva),
    epochs=EPOCHS_K, batch_size=BATCH_K,
    shuffle=True, verbose=2, callbacks=[cb_ckpt])
train_time_keras = time.time() - t0

hist_train_k = [float(v) for v in lich_su.history["loss"]]
hist_val_k   = [float(v) for v in lich_su.history["val_loss"]]
best_epoch_k = int(np.argmin(hist_val_k)) + 1

print()
print(f"Tổng thời gian huấn luyện: {train_time_keras:.2f} giây")
print(f"Epoch tốt nhất theo val_loss: {best_epoch_k} (val_loss = {hist_val_k[best_epoch_k - 1]:.6f})")

model_k.load_weights(ckpt_path)
print("Đã khôi phục bộ trọng số của epoch tốt nhất.")

Epoch 1/40


375/375 - 1s - 4ms/step - loss: 9.6111 - mae: 1.6591 - val_loss: 0.5152 - val_mae: 0.5433


Epoch 2/40


375/375 - 1s - 2ms/step - loss: 0.5021 - mae: 0.5388 - val_loss: 0.4995 - val_mae: 0.5397


Epoch 3/40


...

Epoch 39/40


375/375 - 1s - 2ms/step - loss: 0.4729 - mae: 0.5239 - val_loss: 0.4711 - val_mae: 0.5238


Epoch 40/40


375/375 - 1s - 2ms/step - loss: 0.4728 - mae: 0.5237 - val_loss: 0.4707 - val_mae: 0.5237



Tổng thời gian huấn luyện: 36.35 giây
Epoch tốt nhất theo val_loss: 40 (val_loss = 0.470730)
Đã khôi phục bộ trọng số của epoch tốt nhất.


### Diễn giải nhật ký huấn luyện

Hình dạng đường cong lặp lại đúng ba giai đoạn đã quan sát ở hai notebook trước: sụt mạnh trong vài
epoch đầu, giảm chậm dần ở giai đoạn giữa, rồi gần như đi ngang.

Chênh lệch giữa `loss` và `val_loss` rất nhỏ trong suốt quá trình, và `val_loss` không có xu hướng
tăng trở lại ở cuối. Kết luận về việc mô hình không quá khớp vì vậy nhất quán trên cả ba framework,
điều này quan trọng vì nó cho thấy nhận định không phải hệ quả ngẫu nhiên của một hiện thực cụ thể
mà là tính chất của cặp kiến trúc và dữ liệu.

Một lưu ý khi đọc cột `loss` do Keras in ra: đó là **trung bình trượt của mất mát trên từng lô
trong lúc trọng số đang thay đổi**, chứ không phải mất mát tính lại trên toàn tập huấn luyện sau
khi epoch kết thúc như cách notebook 01 và 02 làm. Ở những epoch đầu, khi trọng số thay đổi nhanh,
con số của Keras sẽ cao hơn một chút; về cuối, khi trọng số ổn định, hai cách tính hội tụ về nhau.
Đây là lý do đường cong huấn luyện của Keras ở hình tổng hợp trông dốc hơn đôi chút trong vùng đầu.

---

## 7. Đánh giá trên tập kiểm tra và ghi kết quả trung gian

In [10]:
pred_test_k = model_k.predict(Xte_k, verbose=0).ravel()
pred_tr_k   = model_k.predict(Xtr_k, verbose=0).ravel()
pred_va_k   = model_k.predict(Xva_k, verbose=0).ravel()

mk_test  = danh_gia_hoi_quy(yte.ravel(), pred_test_k)
mk_train = danh_gia_hoi_quy(ytr.ravel(), pred_tr_k)
mk_val   = danh_gia_hoi_quy(yva.ravel(), pred_va_k)

display(pd.DataFrame([mk_train, mk_val, mk_test],
                     index=["Train", "Validation", "Test"]).round(6))

print()
print("=== CHỈ SỐ TRÊN TẬP KIỂM TRA (TensorFlow/Keras) ===")
print(f"  RMSE (USD)  : {mk_test['rmse_usd']:>14,.2f}")
print(f"  MAE  (USD)  : {mk_test['mae_usd']:>14,.2f}")
print(f"  R^2  (log)  : {mk_test['r2']:>14.6f}")
print(f"  RMSE (log)  : {mk_test['rmse_log']:>14.6f}")
print(f"  MAE  (log)  : {mk_test['mae_log']:>14.6f}")
print(f"  Sai số tương đối trung bình: {(np.exp(mk_test['mae_log']) - 1) * 100:.2f} %")

partial_keras = {
    "framework": "TensorFlow/Keras",
    "params": int(params_k),
    "train_time_s": float(train_time_keras),
    "epochs": int(EPOCHS_K),
    "best_epoch": int(best_epoch_k),
    "rmse_usd": mk_test["rmse_usd"],
    "mae_usd":  mk_test["mae_usd"],
    "r2":       mk_test["r2"],
    "rmse_log": mk_test["rmse_log"],
    "mae_log":  mk_test["mae_log"],
    "loss":     float(hist_val_k[best_epoch_k - 1]),
    "history": {"train_loss": hist_train_k, "val_loss": hist_val_k},
    "scatter_sample": lay_scatter_sample(yte.ravel(), pred_test_k, n=200),
    "dataset": {"file": "usa_real_estate_150k.csv",
                "n_raw": int(n_raw), "n_clean": int(n_clean),
                "n_train": int(n_train), "n_val": int(n_val), "n_test": int(n_test),
                "n_features": 8},
}

with open(REP_DIR + "/_partial_tensorflow.json", "w", encoding="utf-8") as f:
    json.dump(partial_keras, f, ensure_ascii=False, indent=2)
print()
print("Đã ghi:", REP_DIR + "/_partial_tensorflow.json")

,rmse_usd,mae_usd,r2,rmse_log,mae_log
Train,566865.949184,293270.266529,0.397189,0.684403,0.521017
Validation,570293.381826,296994.133848,0.400275,0.686098,0.523680
Test,563725.184955,294268.009226,0.389577,0.689986,0.524653



=== CHỈ SỐ TRÊN TẬP KIỂM TRA (TensorFlow/Keras) ===
  RMSE (USD)  :     563,725.18
  MAE  (USD)  :     294,268.01
  R^2  (log)  :       0.389577
  RMSE (log)  :       0.689986
  MAE  (log)  :       0.524653
  Sai số tương đối trung bình: 68.99 %

Đã ghi: ../reports/_partial_tensorflow.json


### Diễn giải chỉ số

Ba dòng Train, Validation và Test lại nằm rất sát nhau, tiếp tục khẳng định mô hình không quá khớp
và con số trên tập kiểm tra là ước lượng tin cậy.

Chỉ số dễ diễn giải nhất đối với người đọc phổ thông vẫn là sai số tương đối trung bình
$e^{\text{MAE}_{\log}} - 1$, cho biết mức chênh lệch phần trăm điển hình giữa giá dự đoán và giá
thực. Khác với RMSE tính bằng USD, chỉ số này không bị chi phối bởi một nhóm nhỏ bất động sản cao
cấp, nên nó mô tả chất lượng mô hình một cách công bằng giữa các phân khúc giá.

---
---

# PHẦN B. Tổng hợp kết quả ba framework

Từ đây trở đi notebook không huấn luyện thêm mô hình nào. Nhiệm vụ còn lại là đọc ba file trung
gian, dựng ba hình tổng hợp bắt buộc và ghi file metrics cuối cùng.

In [11]:
duong_dan = {
    "numpy":      REP_DIR + "/_partial_numpy.json",
    "pytorch":    REP_DIR + "/_partial_pytorch.json",
    "tensorflow": REP_DIR + "/_partial_tensorflow.json",
}

ket_qua = {}
for khoa, dd in duong_dan.items():
    if not os.path.exists(dd):
        raise FileNotFoundError(
            f"Thiếu file trung gian {dd}. Hãy chạy notebook 01 và 02 trước notebook 03.")
    with open(dd, encoding="utf-8") as f:
        ket_qua[khoa] = json.load(f)
    print(f"Đã nạp {khoa:<11} <- {os.path.basename(dd)}")

NHAN = {"numpy": "NumPy thuần", "pytorch": "PyTorch", "tensorflow": "TensorFlow/Keras"}
MAU  = {"numpy": "#4C72B0", "pytorch": "#DD8452", "tensorflow": "#55A868"}
THU_TU = ["numpy", "pytorch", "tensorflow"]

bang_ss = pd.DataFrame([{
    "Framework": NHAN[k],
    "Tham số": ket_qua[k]["params"],
    "Epoch": ket_qua[k]["epochs"],
    "Epoch tốt nhất": ket_qua[k]["best_epoch"],
    "Thời gian (s)": round(ket_qua[k]["train_time_s"], 2),
    "RMSE (USD)": round(ket_qua[k]["rmse_usd"], 2),
    "MAE (USD)": round(ket_qua[k]["mae_usd"], 2),
    "R^2 (log)": round(ket_qua[k]["r2"], 6),
    "RMSE (log)": round(ket_qua[k]["rmse_log"], 6),
    "MAE (log)": round(ket_qua[k]["mae_log"], 6),
} for k in THU_TU]).set_index("Framework")

display(bang_ss)

Đã nạp numpy       <- _partial_numpy.json
Đã nạp pytorch     <- _partial_pytorch.json
Đã nạp tensorflow  <- _partial_tensorflow.json


,Tham số,Epoch,Epoch tốt nhất,Thời gian (s),RMSE (USD),MAE (USD),R^2 (log),RMSE (log),MAE (log)
Framework,,,,,,,,,
NumPy thuần,1377,40,35,74.55,559141.95,290666.02,0.399226,0.684511,0.518748
PyTorch,1377,40,40,43.36,555235.12,290805.23,0.403889,0.681849,0.518725
TensorFlow/Keras,1377,40,40,36.35,563725.18,294268.01,0.389577,0.689986,0.524653


### Diễn giải bảng so sánh ba framework

Bảng trên là kết quả trung tâm của toàn bộ miền `house_price`. Ba nhận xét quan trọng.

**Về số tham số.** Cả ba cột đều bằng 1 377. Đây là điều kiện tiên quyết để phép so sánh có nghĩa:
ba mô hình cùng năng lực biểu diễn, cùng dữ liệu, cùng thuật toán tối ưu và cùng số epoch.

**Về chất lượng dự đoán.** Chênh lệch $R^2$ giữa ba framework nhỏ hơn nhiều so với khoảng cách giữa
mô hình và đối chứng hằng số đã tính ở notebook 01. Kết luận về chất lượng mô hình vì vậy bền vững
trước lựa chọn framework. Ba nguồn gây chênh lệch còn lại đã được liệt kê ở notebook 02 mục 7: khởi
tạo mặc định khác nhau, thứ tự xáo trộn lô khác nhau, và độ chính xác số học khác nhau
(`float64` ở NumPy so với `float32` ở hai framework kia).

**Về thời gian huấn luyện.** Đây là nơi khác biệt lớn nhất và cũng dễ hiểu nhất. Hiện thực NumPy
thuần phải quay vòng qua trình thông dịch Python cho từng lô, trong khi PyTorch và TensorFlow gọi
xuống các nhân tính toán biên dịch sẵn và tự song song hóa trên nhiều luồng CPU. Cần lưu ý rằng
với mô hình chỉ 1 377 tham số và chuỗi dài 8, phần chi phí cố định cho mỗi lô chiếm tỷ trọng lớn,
nên tỷ số thời gian ở đây phản ánh chi phí điều phối nhiều hơn là hiệu năng tính toán thuần túy.

---

## 8. Hình 1 trên 3: `fig_house_loss_curves.png`

Ba bảng con cạnh nhau, mỗi bảng vẽ mất mát MSE trên thang log theo epoch cho một framework, gồm cả
đường huấn luyện và đường kiểm định, kèm dấu đánh dấu epoch tốt nhất.

In [12]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)

for ax, k in zip(axes, THU_TU):
    r = ket_qua[k]
    ep = np.arange(1, len(r["history"]["train_loss"]) + 1)
    ax.plot(ep, r["history"]["train_loss"], color=MAU[k], linewidth=2, label="Huấn luyện")
    ax.plot(ep, r["history"]["val_loss"], color=MAU[k], linewidth=2,
            linestyle="--", alpha=0.85, label="Kiểm định")
    be = r["best_epoch"]
    ax.axvline(be, color="#C44E52", linestyle=":", linewidth=1.8)
    ax.scatter([be], [r["history"]["val_loss"][be - 1]], color="#C44E52", zorder=5, s=60,
               label="Epoch tốt nhất = {}".format(be))
    ax.set_title("{}\n(MSE kiểm định nhỏ nhất = {:.4f})".format(NHAN[k], min(r["history"]["val_loss"])),
                 fontsize=12, fontweight="bold")
    ax.set_xlabel("Epoch")
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

axes[0].set_ylabel("Mất mát MSE trên thang log_price")
fig.suptitle("Đường cong mất mát MSE theo epoch của ba hiện thực CNN 1 chiều",
             fontsize=14, fontweight="bold")
fig.tight_layout()
fig.savefig(FIG_DIR + "/fig_house_loss_curves.png", dpi=150, bbox_inches="tight")
plt.close(fig)
print("Đã lưu hình:", FIG_DIR + "/fig_house_loss_curves.png")
for k in THU_TU:
    h = ket_qua[k]["history"]
    print("  {:<11} MSE huấn luyện epoch đầu = {:.4f} -> epoch cuối = {:.4f}; "
          "MSE kiểm định nhỏ nhất = {:.4f} tại epoch {}".format(
              NHAN[k], h["train_loss"][0], h["train_loss"][-1],
              min(h["val_loss"]), ket_qua[k]["best_epoch"]))

Đã lưu hình: ../reports/figures/fig_house_loss_curves.png
  NumPy thuần MSE huấn luyện epoch đầu = 0.5735 -> epoch cuối = 0.4636; MSE kiểm định nhỏ nhất = 0.4647 tại epoch 35
  PyTorch     MSE huấn luyện epoch đầu = 0.5356 -> epoch cuối = 0.4593; MSE kiểm định nhỏ nhất = 0.4623 tại epoch 40
  TensorFlow/Keras MSE huấn luyện epoch đầu = 9.6111 -> epoch cuối = 0.4728; MSE kiểm định nhỏ nhất = 0.4707 tại epoch 40


### Diễn giải hình `fig_house_loss_curves.png`

Ba bảng con dùng chung trục tung nên có thể so sánh trực tiếp.

**Về mức hội tụ**, ba đường kết thúc ở những giá trị MSE rất gần nhau, phù hợp với kết luận rút ra
từ bảng so sánh: chất lượng cuối cùng gần như không phụ thuộc framework.

**Về tốc độ hội tụ trong những epoch đầu**, đường của Keras nằm cao hơn hai đường kia. Nguyên nhân
đã giải thích ở mục 6: Keras báo cáo trung bình trượt của mất mát theo lô trong lúc trọng số còn
đang thay đổi nhanh, còn NumPy và PyTorch tính lại mất mát trên toàn tập sau khi epoch kết thúc.
Đây là khác biệt về cách đo, không phải khác biệt về chất lượng học.

**Về khoảng cách giữa hai đường trong mỗi bảng**, đường kiểm định bám sát đường huấn luyện ở cả ba
framework, không có hiện tượng đường kiểm định tách lên trên rồi đi ngược chiều. Đây là dấu hiệu
kinh điển của một mô hình **chưa khai thác hết năng lực** chứ không phải mô hình quá khớp. Nếu muốn
cải thiện, hướng đúng là tăng số kênh hoặc bổ sung đặc trưng về vị trí địa lý, chứ không phải thêm
dropout hay suy giảm trọng số.

**Về vị trí epoch tốt nhất**, cả ba đều nằm ở nửa sau của dải huấn luyện, xác nhận rằng 40 epoch là
lựa chọn hợp lý: đủ dài để hội tụ nhưng chưa tới mức lãng phí.

---

## 9. Hình 2 trên 3: `fig_house_scatter.png`

Ba bảng con tán xạ giữa giá trị thực và giá trị dự đoán trên thang `log_price`, kèm đường chéo
$y = x$ và chú thích hệ số xác định. Dữ liệu vẽ là **đúng 200 điểm** trong khóa `scatter_sample`
của file metrics, và vì cả ba framework dùng chung một hạt giống lấy mẫu nên ba bảng con mô tả
**cùng một tập 200 bất động sản**. Nhờ đó, so sánh giữa ba bảng là so sánh từng điểm một chứ không
phải so sánh hai mẫu ngẫu nhiên khác nhau.

In [13]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5.8), sharex=True, sharey=True)

lo = min(min(ket_qua[k]["scatter_sample"]["y_true"]) for k in THU_TU)
hi = max(max(ket_qua[k]["scatter_sample"]["y_true"]) for k in THU_TU)
bien = [lo - 0.4, hi + 0.4]

for ax, k in zip(axes, THU_TU):
    r = ket_qua[k]
    yt = np.array(r["scatter_sample"]["y_true"])
    yp = np.array(r["scatter_sample"]["y_pred"])
    ax.scatter(yt, yp, s=26, alpha=0.62, color=MAU[k], edgecolor="white", linewidth=0.4,
               label="200 mẫu kiểm tra")
    ax.plot(bien, bien, color="#C44E52", linestyle="--", linewidth=2, label="Đường chéo y = x")
    ax.set_xlim(bien)
    ax.set_ylim(bien)
    ax.set_title("{}\n$R^2$ (toàn tập kiểm tra) = {:.4f}".format(NHAN[k], r["r2"]),
                 fontsize=12, fontweight="bold")
    ax.set_xlabel("log_price thực tế")
    ax.grid(True, alpha=0.3)
    sai_so_mau = float(np.sqrt(np.mean((yp - yt) ** 2)))
    ax.text(0.04, 0.94, "RMSE trên 200 mẫu = {:.4f}".format(sai_so_mau),
            transform=ax.transAxes, fontsize=10, va="top",
            bbox=dict(boxstyle="round", facecolor="white", alpha=0.85, edgecolor="#999999"))
    ax.legend(fontsize=9, loc="lower right")

axes[0].set_ylabel("log_price dự đoán")
fig.suptitle("Giá trị thực so với giá trị dự đoán trên thang logarit (200 mẫu chung cho ba framework)",
             fontsize=14, fontweight="bold")
fig.tight_layout()
fig.savefig(FIG_DIR + "/fig_house_scatter.png", dpi=150, bbox_inches="tight")
plt.close(fig)
print("Đã lưu hình:", FIG_DIR + "/fig_house_scatter.png")

for k in THU_TU:
    yt = np.array(ket_qua[k]["scatter_sample"]["y_true"])
    yp = np.array(ket_qua[k]["scatter_sample"]["y_pred"])
    print("  {:<11} trên 200 mẫu: độ dốc hồi quy = {:.4f}, "
          "độ lệch chuẩn dự đoán = {:.4f} (thực tế = {:.4f})".format(
              NHAN[k], float(np.polyfit(yt, yp, 1)[0]), float(yp.std()), float(yt.std())))

Đã lưu hình: ../reports/figures/fig_house_scatter.png
  NumPy thuần trên 200 mẫu: độ dốc hồi quy = 0.4111, độ lệch chuẩn dự đoán = 0.5713 (thực tế = 0.8477)
  PyTorch     trên 200 mẫu: độ dốc hồi quy = 0.3981, độ lệch chuẩn dự đoán = 0.5511 (thực tế = 0.8477)
  TensorFlow/Keras trên 200 mẫu: độ dốc hồi quy = 0.3626, độ lệch chuẩn dự đoán = 0.5161 (thực tế = 0.8477)


### Diễn giải hình `fig_house_scatter.png`

Ba bảng con cho thấy cùng một dạng hình học, và dạng hình học đó nói lên nhiều điều hơn con số
$R^2$ đơn lẻ.

**Đám mây điểm nghiêng theo đường chéo nhưng dẹt hơn đường chéo.** Đây là hiện tượng **co về trung
bình** (regression to the mean), hệ quả tất yếu của việc cực tiểu hóa sai số bình phương khi tín
hiệu trong đặc trưng còn hạn chế. Con số in kèm bên dưới định lượng điều này: độ dốc của đường hồi
quy giữa dự đoán và thực tế nhỏ hơn 1, và độ lệch chuẩn của dự đoán nhỏ hơn hẳn độ lệch chuẩn của
giá trị thực. Nói cách khác, mô hình **dự đoán quá nhẹ tay**: nó kéo bất động sản rẻ lên và kéo bất
động sản đắt xuống về phía trung bình.

**Hệ quả trực tiếp tới sai số USD.** Vì phần bị kéo xuống nằm ở đuôi phải, nơi hàm mũ khuếch đại
mạnh nhất, những bất động sản đắt tiền bị dự đoán thấp sẽ đóng góp sai số USD rất lớn. Đây chính là
cơ chế đứng sau tỷ số RMSE trên MAE cao đã quan sát ở notebook 01 mục 12.

**Ba bảng con gần như trùng nhau.** Vì cả ba dùng đúng 200 bất động sản giống nhau, ta có thể so
sánh từng điểm. Sự tương đồng cho thấy ba framework không chỉ đạt chỉ số tổng hợp giống nhau mà còn
mắc **cùng những lỗi** trên cùng những mẫu. Điều này quan trọng về mặt phương pháp: nó chứng minh
sai số còn lại đến từ giới hạn của tập đặc trưng chứ không phải từ sự ngẫu nhiên của quá trình tối
ưu. Tám đặc trưng hiện có mô tả kích thước và cấu hình phòng, nhưng hoàn toàn không mang thông tin
về **vị trí địa lý**, vốn là yếu tố quyết định giá bất động sản. Đó là trần hiệu năng thực sự của
bài toán trong cấu hình hiện tại.

---

## 10. Hình 3 trên 3: `fig_house_benchmark.png`

Ba bảng con bar chart cho ba nhóm chỉ số: RMSE tính bằng USD, MAE tính bằng USD, và hệ số xác định
trên thang log. Ba chỉ số này chênh nhau nhiều bậc độ lớn nên không thể vẽ chung một trục tung;
việc tách thành ba bảng con là cách trình bày trung thực duy nhất.

In [14]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5.2))
ten_ft = [NHAN[k] for k in THU_TU]
mau_ft = [MAU[k] for k in THU_TU]

cau_hinh = [
    ("rmse_usd", "RMSE quy đổi về USD", "USD", "{:,.0f}"),
    ("mae_usd",  "MAE quy đổi về USD",  "USD", "{:,.0f}"),
    ("r2",       "Hệ số xác định $R^2$ (thang log)", "$R^2$", "{:.4f}"),
]

for ax, (khoa, tieu_de, nhan_truc, dinh_dang) in zip(axes, cau_hinh):
    gia_tri = [ket_qua[k][khoa] for k in THU_TU]
    thanh = ax.bar(ten_ft, gia_tri, color=mau_ft, edgecolor="white", linewidth=1.2, width=0.62)
    ax.set_title(tieu_de, fontsize=12, fontweight="bold")
    ax.set_ylabel(nhan_truc)
    ax.grid(True, axis="y", alpha=0.3)
    ax.set_ylim(0, max(gia_tri) * 1.18)
    for t, v in zip(thanh, gia_tri):
        ax.text(t.get_x() + t.get_width() / 2, v * 1.02, dinh_dang.format(v),
                ha="center", va="bottom", fontsize=10, fontweight="bold")
    ax.tick_params(axis="x", labelrotation=12)

fig.suptitle("So sánh ba hiện thực CNN 1 chiều trên tập kiểm tra house_price (30 000 mẫu)",
             fontsize=14, fontweight="bold")
fig.tight_layout()
fig.savefig(FIG_DIR + "/fig_house_benchmark.png", dpi=150, bbox_inches="tight")
plt.close(fig)
print("Đã lưu hình:", FIG_DIR + "/fig_house_benchmark.png")

tot_nhat_r2 = max(THU_TU, key=lambda k: ket_qua[k]["r2"])
te_nhat_r2  = min(THU_TU, key=lambda k: ket_qua[k]["r2"])
print()
print("Cao nhất theo R^2 : {} ({:.6f})".format(NHAN[tot_nhat_r2], ket_qua[tot_nhat_r2]["r2"]))
print("Thấp nhất theo R^2: {} ({:.6f})".format(NHAN[te_nhat_r2], ket_qua[te_nhat_r2]["r2"]))
print("Biên độ chênh lệch R^2 giữa ba framework: {:.6f}".format(
    ket_qua[tot_nhat_r2]["r2"] - ket_qua[te_nhat_r2]["r2"]))
print("Biên độ chênh lệch MAE (USD): {:,.2f}".format(
    max(ket_qua[k]["mae_usd"] for k in THU_TU) - min(ket_qua[k]["mae_usd"] for k in THU_TU)))

Đã lưu hình: ../reports/figures/fig_house_benchmark.png

Cao nhất theo R^2 : PyTorch (0.403889)
Thấp nhất theo R^2: TensorFlow/Keras (0.389577)
Biên độ chênh lệch R^2 giữa ba framework: 0.014312
Biên độ chênh lệch MAE (USD): 3,601.99


### Diễn giải hình `fig_house_benchmark.png`

Ba bảng con cho cùng một thông điệp: **ba cột trong mỗi bảng gần bằng nhau**.

Biên độ chênh lệch $R^2$ giữa framework tốt nhất và kém nhất, in ngay dưới hình, nhỏ hơn nhiều bậc
so với chính giá trị $R^2$. Nói cách khác, nếu chỉ nhìn chất lượng dự đoán thì việc chọn NumPy
thuần, PyTorch hay TensorFlow là một quyết định gần như trung tính.

Điều đó dẫn tới nhận định thực tiễn quan trọng nhất của toàn bộ miền `house_price`: **tiêu chí chọn
framework không nằm ở độ chính xác mà nằm ở chi phí phát triển và khả năng mở rộng.** NumPy thuần
có giá trị sư phạm không thể thay thế vì nó buộc người viết phải suy ra từng công thức đạo hàm và
cho phép kiểm tra bằng sai phân hữu hạn, nhưng khối lượng mã lớn hơn nhiều lần và thời gian huấn
luyện dài hơn. PyTorch và TensorFlow xóa bỏ toàn bộ phần đạo hàm thủ công, đổi lại người dùng phải
tin vào thư viện và mất tầm nhìn vào các đại lượng trung gian.

Cần đọc bảng RMSE tính bằng USD với đúng mức thận trọng đã nêu ở notebook 01 mục 12.1. Con số này
chịu ảnh hưởng của khe Jensen và bị khuếch đại ở phân khúc giá cao, nên nó mô tả **tác động kinh
tế** chứ không phải chất lượng thống kê. Chỉ số công bằng giữa các phân khúc giá là $R^2$ và RMSE
trên thang log.

---

## 11. Ghi file metrics cuối cùng

File `reports/metrics_house_price.json` được ghi theo đúng biến thể hồi quy của schema ở Mục 5.3:
bộ chỉ số phân loại được thay bằng `rmse_usd`, `mae_usd`, `r2`, `rmse_log`, `mae_log`, và mỗi mô
hình mang thêm khóa `scatter_sample` chứa 200 cặp giá trị.

In [15]:
KHOA_BAT_BUOC = ["framework", "params", "train_time_s", "epochs", "best_epoch",
                 "rmse_usd", "mae_usd", "r2", "rmse_log", "mae_log", "loss",
                 "history", "scatter_sample"]

models_block = {}
for k in THU_TU:
    r = ket_qua[k]
    models_block[k] = {khoa: r[khoa] for khoa in KHOA_BAT_BUOC}

metrics = {
    "domain": "house_price",
    "task": "regression",
    "dataset": ket_qua["numpy"]["dataset"],
    "models": models_block,
    "notes": (
        "Kiến trúc theo CONTRACT Mục 4: 8 -> Conv1D(16,K=3,same) -> ReLU -> Conv1D(16,K=3,same) "
        "-> ReLU -> MaxPool1D(2) -> Flatten -> Dense(8) -> ReLU -> Dense(1) tuyến tính, mất mát MSE. "
        "Cả ba hiện thực đều có đúng 1377 tham số. RANDOM_SEED=42, chia 64/16/20 phần trăm "
        "(96000/24000/30000) bằng hai lần train_test_split(random_state=42). "
        "SAI LỆCH SO VỚI HỢP ĐỒNG: sau StandardScaler (fit trên train) có thêm một bước winsorize "
        "ở ngưỡng +/-5 độ lệch chuẩn; lý do là bed_bath_prod đạt tới 1833 (hơn 150 độ lệch chuẩn) "
        "và qua mạng sẽ tạo ra dự đoán log_price cực lớn, khiến RMSE quy đổi về USD bị vài chục "
        "bản ghi chi phối và mất ý nghĩa. Bước này chạm tới dưới 1 phần trăm số mẫu. "
        "acre_lot khuyết 32471 giá trị được điền bằng trung vị 0.23 rồi chặn dưới 1e-3 trước khi "
        "lấy logarit. rmse_usd và mae_usd thu được bằng np.exp() trên dự đoán thang log, nên chịu "
        "độ chênh Jensen (E[exp(Z)] > exp(E[Z])) và không được hiệu chỉnh Smearing; r2/rmse_log/mae_log "
        "tính trên thang log. history chỉ có train_loss và val_loss vì đây là bài toán hồi quy, "
        "không có khái niệm accuracy. Huấn luyện 40 epoch, batch 256, Adam lr=3e-3 cho cả ba "
        "framework; chọn epoch theo val_loss, không bao giờ dùng tập test để chọn epoch. "
        "Toàn bộ chạy trên CPU."
    ),
}

out_path = REP_DIR + "/metrics_house_price.json"
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(metrics, f, ensure_ascii=False, indent=2)

print("Đã ghi:", out_path)
print()
print("--- Kiểm tra tính toàn vẹn theo hợp đồng Mục 5.3 ---")
print("domain :", metrics["domain"])
print("task   :", metrics["task"])
print("dataset:", metrics["dataset"])
for k in THU_TU:
    m = metrics["models"][k]
    thieu = [x for x in KHOA_BAT_BUOC if x not in m]
    print(f"  {k:<11} khóa thiếu = {thieu if thieu else 'không'}; "
          f"scatter_sample = {len(m['scatter_sample']['y_true'])}/"
          f"{len(m['scatter_sample']['y_pred'])} điểm; "
          f"history = {len(m['history']['train_loss'])} epoch")
print()
print("Kích thước file:", os.path.getsize(out_path), "byte")

Đã ghi: ../reports/metrics_house_price.json

--- Kiểm tra tính toàn vẹn theo hợp đồng Mục 5.3 ---
domain : house_price
task   : regression
dataset: {'file': 'usa_real_estate_150k.csv', 'n_raw': 150000, 'n_clean': 150000, 'n_train': 96000, 'n_val': 24000, 'n_test': 30000, 'n_features': 8}
  numpy       khóa thiếu = không; scatter_sample = 200/200 điểm; history = 40 epoch
  pytorch     khóa thiếu = không; scatter_sample = 200/200 điểm; history = 40 epoch
  tensorflow  khóa thiếu = không; scatter_sample = 200/200 điểm; history = 40 epoch

Kích thước file: 48013 byte


---

## 12. Kiểm tra danh mục hình bắt buộc

Hợp đồng liệt kê đúng bốn hình cho miền `house_price`. Ô lệnh dưới đây xác nhận cả bốn đều đã nằm
trên đĩa với kích thước hợp lệ.

In [16]:
HINH_BAT_BUOC = ["fig_house_eda.png", "fig_house_loss_curves.png",
                 "fig_house_scatter.png", "fig_house_benchmark.png"]

print(f"{'Tên hình':<32}{'Trạng thái':<12}{'Kích thước (KB)':>16}")
print("-" * 60)
tat_ca_ton_tai = True
for ten in HINH_BAT_BUOC:
    dd = FIG_DIR + "/" + ten
    if os.path.exists(dd):
        print(f"{ten:<32}{'CÓ':<12}{os.path.getsize(dd) / 1024:>16.1f}")
    else:
        print(f"{ten:<32}{'THIẾU':<12}{'-':>16}")
        tat_ca_ton_tai = False
print("-" * 60)
print("Đầy đủ bốn hình bắt buộc:", tat_ca_ton_tai)
assert tat_ca_ton_tai, "Thiếu hình bắt buộc theo hợp đồng Mục 6"

Tên hình                        Trạng thái   Kích thước (KB)
------------------------------------------------------------
fig_house_eda.png               CÓ                      86.9
fig_house_loss_curves.png       CÓ                      85.4
fig_house_scatter.png           CÓ                     173.7
fig_house_benchmark.png         CÓ                      91.3
------------------------------------------------------------
Đầy đủ bốn hình bắt buộc: True


---

## 13. Kết luận chung cho miền `house_price`

Ba notebook của miền `house_price` đã hoàn tất yêu cầu của Assignment 04 đối với dữ liệu bảng ở
dạng bài toán hồi quy.

**Về hiện thực.** Cùng một kiến trúc CNN một chiều gồm 1 377 tham số được xây dựng ba lần bằng ba
công cụ khác nhau. Notebook 01 viết từ định nghĩa toán học, bao gồm cả phần lan truyền ngược suy ra
bằng tay và được xác nhận bằng kiểm tra sai phân hữu hạn trên bốn tầng có tham số. Notebook 02 và
03 dùng PyTorch và TensorFlow. Ví dụ tích chập tính tay được lặp lại ở cả ba notebook và cho cùng
một kết quả với sai lệch bằng không, chứng minh ba hiện thực đồng nhất về ngữ nghĩa.

**Về kiến trúc bám theo bài toán.** Đầu ra tuyến tính kết hợp mất mát MSE không phải lựa chọn tùy
ý mà là hệ quả của nguyên lý hợp lý cực đại dưới giả thiết nhiễu Gauss trên thang log. Notebook 01
mục 9.2 và 9.3 đã chứng minh điều này và chỉ ra cụ thể điều gì hỏng nếu thay bằng sigmoid với BCE
hoặc softmax với cross-entropy.

**Về biến đổi logarit và quy đổi ngược.** Huấn luyện trên `log_price` biến sai số nhân thành sai số
cộng, đưa phân phối mục tiêu về gần chuẩn và làm cho giả thiết đồng nhất phương sai của MSE trở nên
hợp lý, như hình `fig_house_eda.png` minh họa. Nhưng phép quy đổi ngược bằng hàm mũ không bảo toàn
kỳ vọng: do bất đẳng thức Jensen, $\exp$ của ước lượng kỳ vọng trên thang log là ước lượng **trung
vị** chứ không phải kỳ vọng trên thang gốc. Báo cáo cố ý không áp dụng hiệu chỉnh Smearing và ghi
rõ giới hạn đó trong khóa `notes` của file metrics.

**Về so sánh ba framework.** Chênh lệch chất lượng giữa ba hiện thực nhỏ hơn nhiều bậc so với
khoảng cách giữa mô hình và đối chứng hằng số. Khác biệt thực sự nằm ở thời gian huấn luyện và khối
lượng mã phải viết, chứ không nằm ở độ chính xác.

**Về trần hiệu năng.** Hình `fig_house_scatter.png` cho thấy cả ba mô hình đều co dự đoán về phía
trung bình và mắc **cùng những lỗi trên cùng những mẫu**. Đây là bằng chứng rằng phần sai số còn
lại bắt nguồn từ giới hạn của tập tám đặc trưng, vốn mô tả kích thước và cấu hình phòng nhưng không
mang bất kỳ thông tin nào về vị trí địa lý. Hướng cải thiện đúng là bổ sung đặc trưng vị trí, chứ
không phải tăng năng lực mô hình hay thêm chính quy hóa.